In [29]:
!pip install langchain langchain-community faiss-cpu pypdf sentence-transformers langchain-openai langchain-classic langchain-core wikipedia python-docx

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/18.9 MB 14.6 MB/s eta 0:00:02
   ------- -------------------------------- 3.7/18.9 MB 10.5 MB/s eta 0:00:02
   ------------- -------------------------- 6.3/18.9 MB 10.8 MB/s eta 0:00:02
   ----------------- ---------------------- 8.4/18.9 MB 10.5 MB/s eta 0:00:01
   ---------------------- ----------------- 10.5/18.9 MB 10.4 MB/s eta 0:00:01
   -------------------------- ------------- 12.6/18.9 MB 10.4 MB/s eta 0:00:01
   ------------------------------- -------- 14.7/18.9 MB 10.4 MB/s eta 0:00:01
   ------------------------------------ --- 17.0/18.9 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------  18.9/18.9 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 10.1 MB/s  0:00:01
   ---------------------------------------- 0.0/570.8 kB ? eta -:--:-

In [30]:
import os
from datetime import datetime
import docx

from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate 

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun

In [31]:
# Setup the Local LLM (Connecting to http://127.0.0.1:1234)
# We use ChatOpenAI because local API servers mimic the OpenAI API format
llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="not-needed", # Local servers usually don't require an API key
    model="google/gemma-3-4b",
    temperature=0.1
)

In [32]:
# Tool 1: Custom Calculator (Bypassing LLMMathChain issues)
def simple_calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    try:
        # Clean up the string (e.g., replace ^ with ** for Python math)
        expression = expression.replace('^', '**').strip()
        # Evaluate the math expression
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error evaluating math expression: {e}. Make sure to pass a valid math string like '2 + 2'."

calculator_tool = Tool(
    name="Calculator",
    func=simple_calculator,
    description="Useful for when you need to answer questions about math or solve arithmetic problems. Input should be a standard math expression."
)

# Tool 2: Wikipedia Extractor
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wikipedia_tool = Tool(
    name="Wikipedia",
    func=wikipedia.run,
    description="Useful for when you need to extract factual information or summaries from Wikipedia."
)

# Tool 3: Document Creator
def create_document(content: str) -> str:
    """Creates a word document with the given content."""
    doc = docx.Document()
    
    # Adding a basic title and the content
    doc.add_heading('Generated Information', 0)
    doc.add_paragraph(content)
    
    # Ensure the target directory exists
    os.makedirs("generated_docs", exist_ok=True)
    
    # Generate timestamped filename
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"generated_docs/doc_{timestamp}.docx"
    
    doc.save(filename)
    return f"A document has been created and saved as '{filename}'. This document contains the requested information."

doc_creator_tool = Tool(
    name="DocumentCreator",
    func=create_document,
    description=(
        "Useful for when you need to create a document or save information to a file. "
        "The input to this tool should be the complete text content to be saved in the document."
    )
)

In [33]:
# Combine tools into a list
tools = [calculator_tool, wikipedia_tool, doc_creator_tool]

# Define the standard ReAct prompt manually to bypass hub mismatch issues
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

# Create the prompt from the template
prompt = PromptTemplate.from_template(template)

# 1. Create the modern ReAct agent 
agent = create_react_agent(llm, tools, prompt)

# 2. Initialize the AgentExecutor (This is what actually runs the agent loop)
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True # Crucial for smaller/local models
)

In [34]:
query = "Create a document of what is control systems and its applications."
print(f"Running query: {query}\n")

# Pass the query as a dictionary with the key "input"
response = agent_executor.invoke({"input": query})

# The actual answer is stored in the "output" key
print("\nFinal Output:")
print(response["output"])

Running query: Create a document of what is control systems and its applications.



> Entering new AgentExecutor chain...
I need to create a document explaining control systems and their applications. I can use the DocumentCreator tool for this.
Action: DocumentCreator
Action Input: "Control Systems: What are they? Control systems are engineering systems that regulate, command, or automate a process or system. They're used in a vast range of industries to maintain desired conditions, such as temperature, pressure, flow rate, speed, and position. There are two main types of control systems: open-loop and closed-loop. Open-loop systems rely on pre-determined settings without feedback, while closed-loop systems use feedback from sensors to continuously adjust the output and correct for errors. Applications of Control Systems:\n\n*   **Industrial Automation:** Robots, conveyor belts, automated assembly lines – all rely on control systems to perform tasks efficiently and accurately.\n*   *